# Stage 3c — Retrieval Demo (Full)

Same retrieval core as `04-retrieval-demo-rag.ipynb` (hybrid BM25+vector+RRF →
LLM rerank → refusal gate → cited answer), **plus the app's router**
(`rag-system-for-quote-history/src/rag_system/query.py`, ported 1:1) and the
**breadth routes** for cross-offer questions.

```
question
  │
  ├─ contains AG#### ──────────► filtered retrieval (that offer only)
  │      ├─ not found ─────────► refusal, NO fallback to other offers
  │      ├─ price question ────► DETERMINISTIC answer from metadata
  │      │                        (no LLM: preis field, German number format)
  │      └─ else ──────────────► rerank + gate + RAG
  │
  ├─ statistics (wie viele / durchschnitt / ausreißer)
  │      └─► FULL SCAN: LLM reads ALL offers (map → JSON facts),
  │          deterministic CODE reduces (count / mean / outliers).
  │          No retrieval, no top-k cap → complete, exact, 0 LLM in reduce.
  │
  ├─ comparison (vergleiche / unterschiede)
  │      └─► MAP-REDUCE: topic retrieval (top-20, dedup by offer)
  │          → LLM map: one fact line per offer → LLM reduce: comparison.
  │          (= LlamaIndex `tree_summarize`, hand-rolled to keep our
  │            hybrid retrieval pipeline)
  │
  ├─ ambiguous (price/date topic, no year) ─► clarification with chips
  │                                            (all offers: id, datum, preis)
  │
  ├─ aggregation (welche/alle/…) ─► RAG + limitation note
  │
  └─ default ───────────────────► plain RAG
        └─ optional HyDE: LLM writes a hypothetical answer passage,
           embedded as a 3rd RRF arm (weights 0.4/0.3/0.3 vs 0.5/0.5)

Compound questions (≥2 "?" or " und ") keep 5 candidates and use a
stricter refusal threshold (4.0 instead of 5.0).
Citations: verbatim quotes + deterministic page lookup → [AG#### | S. X].


## Setup — Environment, LLM & Index

Same as NB4: load `.env`, OpenAI client (thinking OFF, temp 0),
Chroma collection, Ollama embedding model.

In [ ]:
import os, re, json, time
from pathlib import Path
from dotenv import load_dotenv

# Load .env (walk up: notebooks/ -> 01-submission/ -> final-project/)
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    dotenv_file = path / ".env"
    if dotenv_file.exists():
        load_dotenv(dotenv_file, override=True)
        print(f"✅ Loaded .env from {dotenv_file}")
        break

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "not-needed")
EMBED_BASE_URL = os.getenv("EMBED_BASE_URL", "")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

DEMO_DIR = Path.cwd().parent            # 01-submission
CHROMA_DIR = DEMO_DIR / "data" / "db" / "chroma"
COLLECTION = "offers"

# Retrieval pipeline parameters (same defaults as the app's config.py)
RRF_K = 60
W_VEC, W_BM25 = 0.5, 0.5
RERANK_TOP_N = 10
KEEP = 3
REFUSAL_THRESHOLD = 5.0

# --- LLM (remote vLLM, OpenAI-compatible; thinking OFF, temperature 0) ---
from openai import OpenAI
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def llm_chat(prompt: str) -> str:
    resp = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=4096,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content

# --- Index (loaded from disk, exactly like the app) ---
import chromadb
from llama_index.embeddings.ollama import OllamaEmbedding
from types import SimpleNamespace

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection(COLLECTION)
embed_model = OllamaEmbedding(
    model_name=EMBED_MODEL,
    base_url=EMBED_BASE_URL.replace("/v1", ""),
)
print(f"✅ Index: {collection.count()} chunks in '{COLLECTION}' ({CHROMA_DIR})")
print(f"✅ LLM: {LLM_MODEL} at {LLM_BASE_URL} (thinking off, temp 0)")
print(f"✅ Pipeline: RRF k={RRF_K} w={W_VEC}/{W_BM25} → rerank top-{RERANK_TOP_N} → keep {KEEP} → refusal < {REFUSAL_THRESHOLD}")

## Step 1: Hybrid Search (BM25 + Vector + RRF, optional HyDE arm)

Same core as NB4, extended: `rrf_fuse` accepts a third list
(HyDE vector hits) with its own weight. `hybrid_search` takes an
optional `hyde_passage` parameter.

In [ ]:
from rank_bm25 import BM25Okapi
import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)
from nltk.tokenize import word_tokenize

# --- BM25 corpus: the chunk texts straight from Chroma (no separate artifact) ---
_chroma_res = collection.get(include=["documents", "metadatas"])
node_list = [
    SimpleNamespace(node_id=_id, text=_doc, metadata=dict(_meta or {}))
    for _id, _doc, _meta in zip(
        _chroma_res["ids"], _chroma_res["documents"], _chroma_res["metadatas"]
    )
]
corpus_tokens = [word_tokenize(n.text.lower()) for n in node_list]
bm25 = BM25Okapi(corpus_tokens)
print(f"🔤 BM25 corpus: {len(node_list)} chunks from Chroma")

def vector_search(query, top_n=10, angebot_id=None):
    """Vector search via the same embedding model the index was built with."""
    q_emb = embed_model.get_text_embedding(query)
    where = {"angebot_id": angebot_id} if angebot_id else None
    res = collection.query(query_embeddings=[q_emb], n_results=top_n,
                           where=where,
                           include=["documents", "metadatas", "distances"])
    out = []
    for _id, _doc, _meta, _dist in zip(res["ids"][0], res["documents"][0],
                                       res["metadatas"][0], res["distances"][0]):
        meta = dict(_meta or {})
        meta["vec_score"] = max(0.0, 1.0 - _dist / 2.0)   # L2 -> similarity
        out.append(SimpleNamespace(node_id=_id, text=_doc, metadata=meta))
    return out

def bm25_search(query, top_n=10, angebot_id=None):
    """Keyword search over the indexed chunks (optionally restricted to one offer)."""
    scores = bm25.get_scores(word_tokenize(query.lower()))
    if angebot_id:
        for i, n in enumerate(node_list):
            if n.metadata.get("angebot_id") != angebot_id:
                scores[i] = -1.0
    top_ids = scores.argsort()[::-1][:top_n]
    return [node_list[i] for i in top_ids]

def rrf_fuse(vec_results, bm25_results, w_vec=W_VEC, w_bm25=W_BM25, k=RRF_K, top_n=10,
             hyde_results=None, w_hyde=None):
    """Merge ranked node lists via weighted Reciprocal Rank Fusion.

    Optionally fuses a third list (HyDE: vector hits for a hypothetical
    answer passage) with its own weight.
    """
    scores, node_by_id = {}, {}
    for rank, node in enumerate(vec_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_vec / (k + rank + 1)
        node_by_id[node.node_id] = node
    for rank, node in enumerate(bm25_results):
        scores[node.node_id] = scores.get(node.node_id, 0.0) + w_bm25 / (k + rank + 1)
        node_by_id[node.node_id] = node
    if hyde_results is not None and w_hyde:
        for rank, node in enumerate(hyde_results):
            scores[node.node_id] = scores.get(node.node_id, 0.0) + w_hyde / (k + rank + 1)
            node_by_id[node.node_id] = node
    ranked = sorted(scores, key=scores.get, reverse=True)[:top_n]
    out = []
    for nid in ranked:
        node = node_by_id[nid]
        node.metadata["rrf_score"] = scores[nid]
        out.append(node)
    return out

def hybrid_search(query, top_n=RERANK_TOP_N, angebot_id=None, hyde_passage=None):
    """Vector + BM25, merged with RRF. Returns top_n candidates.

    If `angebot_id` is given (the question names a specific offer), both
    search paths are restricted to that offer — same as the app's query.py.
    If `hyde_passage` is given (HyDE), it is embedded and vector-searched as a
    third RRF arm with weights 0.4/0.3/0.3 (vs 0.5/0.5 without HyDE).
    """
    vec = vector_search(query, top_n=top_n, angebot_id=angebot_id)
    kw = bm25_search(query, top_n=top_n, angebot_id=angebot_id)
    if hyde_passage:
        hy = vector_search(hyde_passage, top_n=top_n, angebot_id=angebot_id)
        return rrf_fuse(vec, kw, w_vec=W_VEC_HYDE, w_bm25=W_BM25_HYDE,
                        w_hyde=W_HYDE, hyde_results=hy, top_n=top_n)
    return rrf_fuse(vec, kw, top_n=top_n)

print("✅ Hybrid search ready (vector + BM25 + RRF, weights 0.5/0.5; HyDE arm optional)")


## Step 2: Rerank, Refusal Gate & Cited Answer

Same as NB4, plus `_pre_ranked` param so the router can pass its
already-gated candidates without re-reranking.

In [ ]:
def _strip_think(text):
    """Remove  blocks (Qwen thinking mode) before parsing."""
    return re.sub(r"\x3cthink.*?\x3c/endthink\x3e", "", text, flags=re.DOTALL).strip()

PAGE_RE = re.compile(r"\[Seite (\d+) von \d+\]")

def _norm(s):
    """Normalize whitespace + quote variants for robust quote matching."""
    s = re.sub(r"\s+", " ", s)
    for a, b in [("\u201e", '"'), ("\u201c", '"'), ("\u201d", '"'),
                 ("\u201a", "'"), ("\u2018", "'"), ("\u2019", "'"),
                 ("\u2026", "...")]:
        s = s.replace(a, b)
    return s.strip()

def page_of_quote(chunk_text, quote):
    """Deterministic page lookup: find the verbatim quote in the chunk text and
    return the number of the LAST '[Seite X von Y]' marker before it.
    Returns None if the quote cannot be located (never invent a page)."""
    nq, nt = _norm(quote), _norm(chunk_text)
    pos = nt.find(nq)
    if pos < 0:
        return None  # quote not verbatim in chunk -> no page, never guess
    pages = [int(m.group(1)) for m in PAGE_RE.finditer(nt) if m.start() < pos]
    return pages[-1] if pages else 1

def citation_label(node, quote=None):
    """Citation: [AG#### | S. X] — page resolved deterministically from the
    page markers that are already inside the chunk text. Falls back to
    [AG####] when the quote cannot be located."""
    oid = node.metadata.get("angebot_id", "unknown offer")
    page = page_of_quote(node.text, quote) if quote else None
    return f"[{oid} | S. {page}]" if page else f"[{oid}]"

def llm_rerank(query, candidates, keep=KEEP):
    """Score hybrid candidates with the LLM (0-10) and return the top `keep`."""
    snippets = []
    for i, node in enumerate(candidates, 1):
        snippets.append(f"[{i}] ({citation_label(node)})\n{node.text[:1200]}")
    numbered = "\n\n".join(snippets)

    prompt = (
        "You are a search reranker. Given a query and numbered text passages, "
        "score each passage 0-10 for how well it ANSWERS the query.\n"
        "10 = directly and fully answers, 5 = partially related, 0 = irrelevant.\n"
        "Return ONLY a JSON object mapping passage number to score, e.g. {\"1\": 8, \"2\": 3}.\n\n"
        f"QUERY: {query}\n\nPASSAGES:\n{numbered}"
    )
    # Retry loop: the remote LLM occasionally returns an empty/malformed
    # completion. Retry up to 3x with a short backoff.
    t0 = time.time()
    scores = {}
    for attempt in range(3):
        text = llm_chat(prompt)
        m = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if m:
            try:
                scores = json.loads(m.group(0))
                break
            except json.JSONDecodeError:
                scores = {}
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    elapsed = time.time() - t0

    scored = []
    for i, node in enumerate(candidates, 1):
        try:
            s = float(scores.get(str(i), 0))
        except (ValueError, TypeError):
            s = 0.0
        node.metadata["rerank_score"] = s
        scored.append(node)
    scored.sort(key=lambda n: n.metadata["rerank_score"], reverse=True)
    print(f"   rerank: {elapsed:.1f}s over {len(candidates)} candidates")
    return scored[:keep]

AG_RE = re.compile(r"\bAG\d{4}\b")

def answer(query, top_n=RERANK_TOP_N, keep=KEEP, _pre_ranked=None):
    """Full grounded pipeline: hybrid -> rerank -> (refuse if low) -> cited answer.

    If the question names a specific offer (AG####), retrieval is restricted
    to that offer via a metadata filter — same as the app's query.py.
    Returns (answer_text, ranked_nodes, top_score).
    """
    m = AG_RE.search(query)
    angebot_id = m.group(0) if m else None
    if _pre_ranked is not None:
        # Router already ran hybrid search + rerank + gate.
        ranked = _pre_ranked
    else:
        ranked = llm_rerank(query, candidates=hybrid_search(query, top_n=top_n, angebot_id=angebot_id), keep=keep)
    top_score = ranked[0].metadata.get("rerank_score", 0.0) if ranked else 0.0

    if _pre_ranked is None and top_score < REFUSAL_THRESHOLD:
        msg = (f"Die Angebote enthalten dazu keine verlässliche Antwort "
               f"(bester Kandidat: {top_score:.0f}/10).")
        return msg, ranked, top_score

    context = [f"[{i}] ({citation_label(node)})\n{node.text}"
               for i, node in enumerate(ranked, 1)]
    numbered_context = "\n\n".join(context)

    prompt = (
        "Du bist ein präziser RAG-Assistent für Angebote eines Filmstudios. "
        "Beantworte AUSSCHLIESSLICH anhand der nummerierten Kontexte.\n"
        "Regeln:\n"
        "1. Die Antwort kann in MEHREREN Chunks liegen. Gliedere nach Teilaspekt "
        "(ein kurzer Absatz oder Bullet pro Aspekt).\n"
        "2. Zitiere NACH JEDEM Aspekt inline die Quelle in eckigen Klammern mit "
        "der Angebots-ID — z. B. '...Zahlung innerhalb von 14 Tagen [AG0006].' "
        "Keine Zitatliste am Ende.\n"
        "3. Belege WÖRTLICH: Führe die entscheidende Passage aus dem Kontext "
        "exakt so an, wie sie dort steht, in deutschen Anführungszeichen "
        "(\u201e...\u201c).\n"
        "4. Ordne jede Aussage dem Chunk zu, der sie tatsächlich enthält.\n"
        "5. Kein externes Wissen. Was nicht abgedeckt ist, sagst du: 'Nicht in den Angeboten enthalten.'\n"
        "6. Antworte auf Deutsch.\n\n"
        f"KONTEXT:\n{numbered_context}\n\n"
        f"FRAGE: {query}\n\nANTWORT:"
    )
    answer_text = _strip_think(llm_chat(prompt)).strip()
    answer_text = upgrade_citations(answer_text, ranked)
    return answer_text, ranked, top_score

QUOTE_RE = re.compile(r"\u201e(.+?)\u201c", flags=re.DOTALL)
CITE_RE = re.compile(r"\[(AG\d{4})(?:\s*\|[^\]]*)?\]")

def upgrade_citations(answer_text, ranked):
    """Post-processing: for every [AG####] citation, find the verbatim quote
    that belongs to it in that offer's chunks and append the page:
    [AG####] -> [AG#### | S. X]. Deterministic — no LLM involved.
    Citations that cannot be resolved are left as [AG####]."""
    quotes = [m.group(1).strip() for m in QUOTE_RE.finditer(answer_text)]
    if not quotes:
        return answer_text
    # map offer id -> chunk texts (only the kept, high-scoring nodes)
    texts = {}
    for node in ranked:
        texts.setdefault(node.metadata.get("angebot_id"), []).append(node.text)
    # pair each citation with the nearest preceding quote
    pairs = []  # (cite_match, quote)
    qpos = [(m.start(), m.group(1).strip()) for m in QUOTE_RE.finditer(answer_text)]
    for m in CITE_RE.finditer(answer_text):
        preceding = [q for p, q in qpos if p < m.start()]
        if not preceding:
            continue
        quote = preceding[-1]
        oid = m.group(1)
        page = None
        for t in texts.get(oid, []):
            page = page_of_quote(t, quote)
            if page:
                break
        if page:
            pairs.append((m, f"[{oid} | S. {page}]"))
    # replace from the end so positions stay valid
    for m, repl in reversed(pairs):
        answer_text = answer_text[:m.start()] + repl + answer_text[m.end():]
    return answer_text


## Step 3: HyDE (Hypothetical Document Embeddings)

One extra LLM call: write a hypothetical answer passage → embed →
fuse as a 3rd RRF arm (0.4/0.3/0.3). Fixes embedding mismatch for
paraphrased questions. Switch: `HYDE_ENABLED = False` → plain 0.5/0.5.

In [ ]:
# =====================================================================
# HyDE — Hypothetical Document Embeddings (optional, 3rd RRF arm)
# =====================================================================
# For paraphrased questions the query words rarely match the document
# words (failure mode #2: embedding mismatch). HyDE: the LLM writes a
# short hypothetical answer passage; we embed it and fuse it as a third
# RRF arm. One extra LLM call, no reindexing.

HYDE_ENABLED = True          # switch: False -> plain 0.5/0.5 hybrid
W_VEC_HYDE, W_BM25_HYDE, W_HYDE = 0.4, 0.3, 0.3

def generate_hyde_passage(question: str) -> str:
    """One LLM call: a short passage as the answer would be phrased
    inside a post-production service offer. Thinking off, temp 0."""
    prompt = (
        "Schreibe einen kurzen, sachlichen Absatz (2-3 Saetze), wie die "
        "Antwort auf die folgende Frage in einem Post-Production-Leistungs-"
        "angebot formuliert waere. Nutze typische Fachbegriffe aus solchen "
        "Angeboten (z. B. Zahlungsziel, Skonto, Lieferzeit, Leistungsumfang, "
        "Nettobetrag). Antworte NUR mit dem Absatz, ohne Einleitung.\n\n"
        f"Frage: {question}"
    )
    return _strip_think(llm_chat(prompt)).strip()

def hybrid_search_hyde(query, top_n=RERANK_TOP_N, angebot_id=None):
    """Hybrid search with the optional HyDE arm (3rd RRF list)."""
    if not HYDE_ENABLED:
        return hybrid_search(query, top_n=top_n, angebot_id=angebot_id)
    passage = generate_hyde_passage(query)
    print(f"   hyde: {passage[:120]}...")
    return hybrid_search(query, top_n=top_n, angebot_id=angebot_id,
                         hyde_passage=passage)

print(f"✅ HyDE {'ON' if HYDE_ENABLED else 'OFF'} "
      f"(weights {W_VEC_HYDE}/{W_BM25_HYDE}/{W_HYDE} when on)")


## Why Breadth Routes? (and where HyDE fits)

Cross-offer questions ("wie viele Angebote …", "vergleiche … über alle
Angebote") break the rerank paradigm: the reranker asks *"does THIS chunk
answer the question?"* — but breadth questions need **all** matching offers,
and top-k retrieval is capped by construction (with top-20 you can never
count more than 20). Similarity is not a completeness criterion.

**Statistics route** (count / mean / outliers): full scan — the LLM reads
every offer once (map → structured JSON), then **deterministic code**
reduces (count, percent, mean, outliers). No retrieval, no top-k cap, no LLM
in the reduce → exact numbers, complete coverage (every offer read).
Cost: one LLM call per offer — acceptable at this corpus size; at 5,000
documents this would switch to a vector pre-filter with score threshold +
verification (roadmap, Phase 2).

**Comparison route** (tree_summarize principle): topic retrieval (top-20,
dedup by offer) → LLM map: one fact line per offer → LLM reduce: the
comparison. Hand-rolled (≈25 lines) instead of LlamaIndex's built-in
`tree_summarize` so we keep our hybrid BM25+vector+RRF retrieval.

**HyDE** (Hypothetical Document Embeddings): for paraphrased questions the
query words rarely match the document words (failure mode #2 in the course
video). The LLM writes a short hypothetical answer passage; it is embedded
and fused as a **third RRF arm** (0.4/0.3/0.3 vs 0.5/0.5 without HyDE).
One extra LLM call, no reindexing.


## Step 4: Breadth Routes (code)

**Statistics:** full scan (map → JSON facts per offer) + deterministic code reduce.
**Comparison:** topic retrieval → LLM map (one line per offer) → LLM reduce.

In [ ]:
# =====================================================================
# Breadth routes — statistics (full scan + code reduce) and
# comparison (map-reduce over topic retrieval)
# =====================================================================

STATISTICS_RE = re.compile(
    r"\b(wie viele|wieviel|durchschnitt|ausreißer|ausreisser|ausrreisser|"
    r"anteil|prozent|median|häufig|hoeufig)\b", re.IGNORECASE)
COMPARISON_RE = re.compile(
    r"\b(vergleiche|vergleich|unterschiede|unterschied|unterscheiden|"
    r"unterschiedlich|nebeneinander)\b", re.IGNORECASE)

def is_statistics(q):
    """Count/mean/outlier question -> needs FULL coverage, not top-k."""
    return bool(STATISTICS_RE.search(q))

def is_comparison(q):
    """Cross-offer comparison -> map-reduce over topic retrieval."""
    return bool(COMPARISON_RE.search(q))

# ---------- statistics: full scan (map) + deterministic code (reduce) ----------

def _offer_texts():
    """All chunk texts grouped by offer (complete corpus, no retrieval)."""
    by_offer = {}
    for n in node_list:
        oid = n.metadata.get("angebot_id")
        if oid:
            by_offer.setdefault(oid, []).append(n)
    return by_offer

FACTS_PROMPT = (
    "Du liest ein Angebot eines Filmstudios. Extrahiere NUR Fakten, die "
    "explizit im Text stehen. Antworte NUR mit einem JSON-Objekt dieser Form:\n"
    '{"zahlungsziel_tage": int|null, "skonto_prozent": float|null, '
    '"lieferzeit": string|null, "garantie": string|null, "leistungen": [string]}\n'
    "Regeln:\n"
    "- zahlungsziel_tage: Tage bis zur Zahlung (z. B. 14). 'netto 30 Tage' -> 30. "
    "0 ist KEIN gueltiger Wert. Wenn kein Zahlungsziel genannt wird, MUSS null "
    "stehen (nicht 0, nicht 30 als Standard).\n"
    "- skonto_prozent: Skontoprozent (z. B. 5.0), sonst null.\n"
    "- lieferzeit: kurze Wiedergabe der Lieferfrist, sonst null.\n"
    "- garantie: kurze Wiedergabe der Garantie, sonst null.\n"
    "- leistungen: Liste der genannten Leistungen (z. B. 'Color Grading').\n\n"
    "TEXT:\n{text}"
)

def _extract_offer_facts(text):
    """One LLM call per offer -> structured facts (the MAP step)."""
    prompt = FACTS_PROMPT.replace("{text}", text[:12000])
    for attempt in range(3):
        raw = llm_chat(prompt)
        m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except json.JSONDecodeError:
                pass
        if attempt < 2:
            time.sleep(0.5 * (attempt + 1))
    return None

def full_scan_facts():
    """MAP: the LLM reads EVERY offer once -> structured facts.

    Complete coverage by construction (no retrieval, no top-k cap).
    """
    by_offer = _offer_texts()
    facts, failed = {}, []
    t0 = time.time()
    for i, (oid, nodes) in enumerate(sorted(by_offer.items()), 1):
        text = "\n\n".join(n.text for n in nodes)
        preis = next((n.metadata.get("preis") for n in nodes
                      if n.metadata.get("preis") is not None), None)
        datum = next((n.metadata.get("datum") for n in nodes
                      if n.metadata.get("datum")), None)
        f = _extract_offer_facts(text)
        if f is None:
            failed.append(oid)
            continue
        f["preis"] = preis
        f["datum"] = datum
        facts[oid] = f
        print(f"   [{i}/{len(by_offer)}] {oid}: "
              f"zz={f.get('zahlungsziel_tage')} skonto={f.get('skonto_prozent')}")
    print(f"   full scan: {len(facts)}/{len(by_offer)} offers in "
          f"{time.time() - t0:.0f}s" + (f" (failed: {failed})" if failed else ""))
    return facts

def reduce_statistics(facts):
    """REDUCE: deterministic CODE — count / percent / mean / outliers.

    No LLM: statistics questions need exact numbers, and an LLM that
    'reads 47 partial answers' estimates instead of computing.
    """
    total = len(facts)
    with_zz = {o: f for o, f in facts.items()
               if (f.get("zahlungsziel_tage") or 0) > 0}
    short = {o: f for o, f in with_zz.items() if f["zahlungsziel_tage"] < 30}
    prices = [f["preis"] for f in short.values() if f["preis"] is not None]
    lines = [f"**{len(short)} von {total} Angeboten** haben ein Zahlungsziel "
             f"von weniger als 30 Tagen "
             f"({len(with_zz)} legen überhaupt ein Zahlungsziel fest)."]
    if short:
        detail = ", ".join(f"{o} ({f['zahlungsziel_tage']} Tage)"
                           for o, f in sorted(short.items()))
        lines.append(f"- Angebote: {detail}")
    if prices:
        mean = sum(prices) / len(prices)
        lines.append(f"- Durchschnittlicher Nettobetrag dieser Angebote: "
                     f"**{format_price(mean)}** (n={len(prices)})")
        lines.append(f"- Spanne: {format_price(min(prices))} – "
                     f"{format_price(max(prices))}")
        if len(prices) >= 3:
            sd = (sum((p - mean) ** 2 for p in prices) / len(prices)) ** 0.5
            outliers = {o: f["preis"] for o, f in short.items()
                        if f["preis"] is not None and sd > 0
                        and f["preis"] > mean + 2 * sd}
            if outliers:
                lines.append("- Ausreißer (> Mittelwert + 2σ): " +
                             ", ".join(f"{o} ({format_price(p)})"
                                       for o, p in sorted(outliers.items())))
            else:
                lines.append("- Kein Ausreißer (> Mittelwert + 2σ).")
    lines.append(f"\n*Basis: vollständiger Scan aller {total} Angebote "
                 "(kein Retrieval, keine Top-k-Grenze).*")
    return "\n".join(lines)

def statistics_route(question):
    """Statistics question -> full scan (map) + code reduce. No retrieval."""
    facts = full_scan_facts()
    if not facts:
        return ("Statistics", "Keine Fakten konnten extrahiert werden.", [])
    return ("Statistics (full scan + code reduce)", reduce_statistics(facts), [])

# ---------- comparison: map-reduce over topic retrieval ----------

def comparison_route(question, top_offers=15):
    """MAP-REDUCE (tree_summarize principle) over topic retrieval.

    Retrieve on the topic (top-20, dedup by offer) -> MAP: LLM writes one
    fact line per offer -> REDUCE: LLM compares the lines. Hand-rolled
    instead of LlamaIndex's built-in tree_summarize so we keep our hybrid
    BM25+vector+RRF retrieval pipeline.
    """
    chunks = hybrid_search_hyde(question, top_n=20)
    by_offer = {}
    for n in chunks:
        oid = n.metadata.get("angebot_id")
        if oid:
            by_offer.setdefault(oid, []).append(n)
    offers = list(by_offer.items())[:top_offers]
    print(f"   map: {len(offers)} distinct offers from {len(chunks)} chunks")
    lines = []
    for oid, nodes in offers:
        text = "\n\n".join(n.text for n in nodes)[:2500]
        prompt = (f"Zusammenfassung in EINER Zeile: Was sagt dieses Angebot zu: "
                  f"{question}\nText: {text}\n"
                  f"Antworte NUR mit der Zeile, beginnend mit '{oid}: '")
        lines.append(_strip_think(llm_chat(prompt)).strip())
    prompt = (
        "Vergleiche die folgenden Zeilen zu den Angeboten und schreibe eine "
        "kurze Vergleichszusammenfassung auf Deutsch. Zitiere nach jeder "
        "Aussage die Angebots-ID in eckigen Klammern, z. B. [AG1001]. "
        "Nenne Gemeinsamkeiten und Unterschiede. Kein externes Wissen.\n\n"
        "ZEILEN:\n" + "\n".join(lines) + "\n\nVERGLEICH:"
    )
    text = _strip_think(llm_chat(prompt)).strip()
    ranked = [nodes[0] for _, nodes in offers]
    return ("Map-Reduce (comparison)", text, ranked)

print("✅ Breadth routes ready (statistics: full scan + code reduce; "
      "comparison: map-reduce)")


## Step 5: The Router

Ported 1:1 from the app's `query.py`, extended with the breadth routes.
Order matters: ID → statistics → comparison → ambiguous → aggregation → default.

In [ ]:
# =====================================================================
# The router — ported 1:1 from the app's src/rag_system/query.py,
# extended with the breadth routes (statistics, comparison)
# =====================================================================

AMBIGUOUS_TOPIC_RE = re.compile(
    r"\b(preis|netto|brutto|nettobetrag|gesamtpreis|summe|zahlung|"
    r"zahlungsbedingung|skonto|f[aä]llig|faellig|datum|termin|"
    r"honorar|kosten|preisbasis)\b", re.IGNORECASE)
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")
PRICE_TOPIC_RE = re.compile(
    r"\b(preis|netto|brutto|nettobetrag|bruttopreis|gesamtpreis|summe|"
    r"honorar|kosten)\b", re.IGNORECASE)
AGGREGATION_RE = re.compile(
    r"\b(welche|alle|wie viele|mehr als|mindestens|gr[oö]ßer als|"
    r"groesser als|teurer als|g[uü]nstiger als|im jahr|in dem jahr)\b",
    re.IGNORECASE)

def is_ambiguous(q):
    """Price/date/term question without offer reference or year."""
    return bool(AMBIGUOUS_TOPIC_RE.search(q)) and not YEAR_RE.search(q)

def is_price_question(q):
    return bool(PRICE_TOPIC_RE.search(q))

def is_aggregation(q):
    return bool(AGGREGATION_RE.search(q))

def is_compound(q):
    """Several parts (>=2 '?' or ' und ') -> lenient gate (keep 5, thr 4.0)."""
    return q.count("?") >= 2 or bool(re.search(r"\bund\b", q, re.IGNORECASE))

def format_price(value):
    """German number format: 1.251,03 EUR."""
    if value is None:
        return ""
    return f"{value:,.2f} \u20ac".replace(",", "\u00a0").replace(".", ",").replace("\u00a0", ".")

def price_answer(offer_id, chunks):
    """Deterministic price answer from chunk metadata (no LLM)."""
    for node in chunks:
        price = node.metadata.get("preis")
        if price is not None:
            datum = node.metadata.get("datum") or ""
            date_part = f" (Datum: {datum})" if datum else ""
            return (f"Der Nettobetrag von Angebot **{offer_id}** beträgt "
                    f"**{format_price(price)}**{date_part}.")
    return None

def _rerank_and_gate(question, chunks, compound=False):
    """Rerank + refusal gate. Compound: keep 5, threshold 4.0; else keep 3, 5.0."""
    keep = 5 if compound else KEEP
    threshold = 4.0 if compound else REFUSAL_THRESHOLD
    ranked = llm_rerank(question, chunks, keep=keep)
    top = ranked[0].metadata.get("rerank_score", 0.0) if ranked else 0.0
    if top < threshold:
        msg = ("Ich konnte keine zuverlässige Antwort in den vorliegenden "
               f"Angeboten finden (beste Übereinstimmung: {top:.0f}/10).")
        return ranked, msg
    return ranked, None

def _offer_count():
    """Distinct offers in the index (not chunks)."""
    return len({m.get("angebot_id") for m in collection.get(include=["metadatas"])["metadatas"] if m})

def _clarify(question):
    """Deduplicated offer candidates for clarification chips (no LLM)."""
    q_emb = embed_model.get_text_embedding(question)
    res = collection.query(query_embeddings=[q_emb], n_results=10,
                           include=["metadatas"])
    seen, cands = set(), []
    for i, _id in enumerate(res["ids"][0]):
        md = res["metadatas"][0][i] or {}
        oid = md.get("angebot_id") or _id
        if oid in seen:
            continue
        seen.add(oid)
        cands.append({"angebot_id": oid,
                      "datum": md.get("datum") or "\u2014",
                      "preis": md.get("preis")})
    if not cands:
        return None
    lines = " \u00b7 ".join(
        f"{c['angebot_id']} ({c['datum']}"
        + (f", {format_price(c['preis'])}" if c["preis"] else "") + ")"
        for c in cands)
    return (f"Insgesamt liegen {_offer_count()} Angebote vor \u2014 "
            f"meinst du eines dieser? {lines}"), cands

def run_query(question):
    """The full router. Returns (route, text, ranked_nodes)."""
    m = AG_RE.search(question)
    offer_id = m.group(0) if m else None

    # 1) ID-aware retrieval: filter to the referenced offer, NO fallback.
    if offer_id:
        chunks = hybrid_search(question, top_n=RERANK_TOP_N, angebot_id=offer_id)
        if not chunks:
            return ("Refusal",
                    f"Angebot **{offer_id}** wurde nicht in den vorliegenden "
                    "Angeboten gefunden.", [])
        if is_price_question(question):
            a = price_answer(offer_id, chunks)
            if a:
                return ("RAG (deterministic)", a, chunks)
        ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
        if refusal:
            return ("Refusal", refusal, ranked)
        text, _, _ = answer(question, top_n=RERANK_TOP_N, keep=5 if is_compound(question) else KEEP,
                            _pre_ranked=ranked)
        return ("RAG", text, ranked)

    # 2) Statistics (count/mean/outliers) -> FULL SCAN + code reduce.
    #    Checked BEFORE ambiguous/aggregation: "wie viele ... Zahlungsziel"
    #    would otherwise hit the clarification or RAG path.
    if is_statistics(question):
        return statistics_route(question)

    # 3) Comparison (vergleiche/unterschiede) -> map-reduce over topic retrieval.
    if is_comparison(question):
        return comparison_route(question)

    # 4) Ambiguous price/date/term question -> clarification (no LLM call).
    if is_ambiguous(question):
        c = _clarify(question)
        if c:
            return ("Clarify", c[0], [])

    # 5) Aggregation (welche/alle) -> RAG + explicit limitation note.
    if is_aggregation(question):
        chunks = hybrid_search_hyde(question, top_n=RERANK_TOP_N)
        ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
        if refusal:
            return ("Refusal", refusal, ranked)
        text, _, _ = answer(question, top_n=RERANK_TOP_N,
                            keep=5 if is_compound(question) else KEEP, _pre_ranked=ranked)
        text += ("\n\n*Hinweis: Ich kann nur die hier geladenen Angebote "
                 "vergleichen \u2014 eine vollständige Auswertung über alle "
                 "Angebote folgt mit dem SQL-Pfad.*")
        return ("RAG", text, ranked)

    # 6) Default: grounded RAG (with optional HyDE arm).
    chunks = hybrid_search_hyde(question, top_n=RERANK_TOP_N)
    ranked, refusal = _rerank_and_gate(question, chunks, is_compound(question))
    if refusal:
        return ("Refusal", refusal, ranked)
    text, _, _ = answer(question, top_n=RERANK_TOP_N,
                        keep=5 if is_compound(question) else KEEP, _pre_ranked=ranked)
    return ("RAG", text, ranked)


## Step 6: Smoke Test (7 questions)

One question per router path: deterministic price, refusal, clarify,
aggregation, compound RAG, statistics (full scan), comparison (map-reduce).

In [ ]:
# =====================================================================
# Smoke test — one question per router path
# =====================================================================
SMOKE = [
    # 1) ID + price -> deterministic answer from metadata (7.380,00 EUR)
    "Wie hoch war der Preis in AG1001?",
    # 2) ID, not found -> refusal WITHOUT fallback
    "Wie sind die Zahlungsbedingungen in AG9999 geregelt?",
    # 3) ambiguous price question (no year, no ID) -> clarification chips
    "Wie hoch war der Preis?",
    # 4) aggregation -> RAG + SQL-path limitation note
    "Welche Angebote sind im Jahr 2024?",
    # 5) default RAG (compound: ' und ' -> keep 5, threshold 4.0)
    "Wie ist die Abnahme geregelt und welche Formate werden beim "
    "Mastering geliefert?",
    # 6) statistics -> full scan (map) + code reduce (exact, complete)
    "Wie viele unserer Angebote haben ein Zahlungsziel von weniger als "
    "einem Monat? Wie hoch ist der durchschnittliche Betrag, und gibt es "
    "Ausreißer?",
    # 7) comparison -> map-reduce over topic retrieval
    "Vergleiche die Color Grading Leistungen über alle Angebote.",
]

for q in SMOKE:
    print("=" * 70)
    print("Q:", q)
    print("-" * 70)
    route, text, ranked = run_query(q)
    print(f"ROUTE: {route}")
    if ranked:
        for n in ranked[:3]:
            md = n.metadata
            print(f"  {md.get('angebot_id')} | {md.get('datum')} | "
                  f"rerank={md.get('rerank_score', 0):.0f}")
    print()
    print(text)
    print()
